        # Exercise 3 - Cost breakdowns, cash-flow diagrams, and sensitivity analysis

        [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea-workshop/blob/main/notebooks/03_cost_breakdowns_and_sensitivity.ipynb)

        Totals hide where the money goes. This notebook uses OpenPyTEA's analysis and plotting functions to break costs
down, draw the cumulative cash-flow diagram, and ask *"what if"* questions with one-way sensitivity curves and
tornado plots.

**Time in the session: 15 minutes** (Exercise 3 of 6). Bonus tasks are for participants who finish early; skip them if you are behind, the solutions are in the notebook.

        ## Learning goals

        - Apply the `*_data()` + `plot_*()` pattern that every OpenPyTEA visualization follows.
- Produce stacked-bar breakdowns of direct costs, fixed capital, variable and fixed OPEX, and levelized cost, for one plant or several side by side.
- Read a cumulative cash-flow diagram: maximum investment, break-even point, final cumulative value.
- Run a one-way sensitivity analysis on a price, a financial assumption, or a process quantity, and read the break-even point off the curve.
- Rank cost drivers with a tornado plot and know its limitations.

        ## Background references

        - Tamarona, P.B., Vlugt, T.J.H., & Ramdin, M. (2026). OpenPyTEA: An open-source python toolkit for techno-economic assessment of chemical process plants and energy systems with economic sensitivity and uncertainty evaluation. *SoftwareX, 35*, 102816. https://doi.org/10.1016/j.softx.2026.102816
- Van der Spek, M., Roussanaly, S., & Rubin, E.S. (2019). Best practices and recent advances in CCS cost engineering and economic analysis. *IJGGC, 83*, 91-104. https://doi.org/10.1016/j.ijggc.2019.02.006
- OpenPyTEA documentation: https://openpytea.readthedocs.io

## 0) Setup: rebuild the green-ammonia plant from notebooks 01 and 02

In [ ]:
# Run this cell first. It installs OpenPyTEA if it is missing (for example on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
from openpytea import Equipment

# Equipment of the green-ammonia synthesis loop (sizes from the process simulation, see notebook 01)
compressor  = Equipment(name="C-101 Syngas compressor", param=1_200, process_type="Fluids",
                        category="Compressors, fans, & blowers", type="Compressor, centrifugal",
                        material="Carbon steel")
motor       = Equipment(name="M-101 Compressor motor", param=1_200 / 0.90, process_type="Electrical",
                        category="Motors & generators", type="Totally enclosed motor")
exchanger   = Equipment(name="E-101 Feed/effluent exchanger", param=250, process_type="Fluids",
                        category="Heat Exchangers", type="U-tube shell & tube",
                        material="304 stainless steel")
converter   = Equipment(name="R-101 Ammonia converter", param=12_000, process_type="Fluids",
                        category="Reactors", type="Tubular fixed bed", material="304 stainless steel")
separator   = Equipment(name="V-101 Ammonia separator", param=25, process_type="Fluids",
                        category="Pressure vessels", type="Vertical", material="Carbon steel")
chiller     = Equipment(name="K-101 Refrigeration unit", param=800, process_type="Fluids",
                        category="Utilities", type="Packaged mechanical refrigerator")
product_pump = Equipment(name="P-101 Product pump", param=20, process_type="Fluids",
                         category="Pumps", type="Single-stage centrifugal", material="Carbon steel")

equipment_list = [compressor, motor, exchanger, converter, separator, chiller, product_pump]

In [ ]:
from openpytea import Plant

ammonia_config = {
    "plant_name": "Green Ammonia",
    "process_type": "Fluids",
    "country": "Netherlands",
    "equipment": equipment_list,

    # Financial assumptions
    "interest_rate": 0.08,
    "project_lifetime": 20,      # years
    "plant_utilization": 0.90,   # fraction of the year on stream
    "tax_rate": 0.25,

    # Products: the first entry is the main product used for the levelized cost
    "plant_products": {
        "ammonia": {"production": 100_000, "price": 0.95},   # kg/day, USD/kg
    },

    # Variable OPEX: daily consumption and unit price
    "variable_opex_inputs": {
        "hydrogen":    {"consumption": 17_800, "price": 2.00},   # kg/day,  USD/kg
        "nitrogen":    {"consumption": 82_400, "price": 0.05},   # kg/day,  USD/kg
        "electricity": {"consumption": 60_000, "price": 0.08},   # kWh/day, USD/kWh
    },
}

ammonia_plant = Plant(ammonia_config)
ammonia_plant.calculate_all()
print(f"Fixed capital investment : {ammonia_plant.fixed_capital:,.0f} USD")
print(f"Levelized cost of ammonia: {ammonia_plant.levelized_cost:.3f} USD/kg")
print(f"Net present value        : {ammonia_plant.npv:,.0f} USD")

## 1) The data + plot pattern

Every visualization in OpenPyTEA is two functions: a `*_data()` function in `openpytea.analysis` that returns a
plain dictionary, and a `plot_*()` function in `openpytea.plotting` that draws it with matplotlib. You can
therefore reuse the numbers in a report or in your own plotting code.

| Data function | Plot function | Shows |
|---|---|---|
| `direct_costs_data` | `plot_stacked_bar` | direct cost per equipment item |
| `fixed_capital_data` | `plot_stacked_bar` | ISBL, OSBL, D&E, contingency (and additional CAPEX) |
| `variable_opex_data` | `plot_stacked_bar` | variable OPEX per feed and utility |
| `fixed_opex_data` | `plot_stacked_bar` | fixed OPEX components |
| `levelized_cost_data` | `plot_stacked_bar` | LCOP split into discounted CAPEX, OPEX, and by-product credit |
| `cash_flow_data` | `plot_cash_flow` | cumulative cash flow over the project life |
| `sensitivity_data` | `plot_sensitivity` | one-way sensitivity curve |
| `tornado_data` | `plot_tornado` | one-at-a-time ranking of parameters |
| `monte_carlo` | `plot_monte_carlo`, `plot_multiple_monte_carlo`, `plot_monte_carlo_inputs` | notebook 04 |

In [ ]:
from openpytea.analysis import (direct_costs_data, fixed_capital_data, variable_opex_data,
                                fixed_opex_data, levelized_cost_data, cash_flow_data,
                                sensitivity_data, tornado_data)
from openpytea.plotting import plot_stacked_bar, plot_cash_flow, plot_sensitivity, plot_tornado
import matplotlib.pyplot as plt

## 2) Cost breakdowns

In [ ]:
fig, ax = plot_stacked_bar(direct_costs_data(ammonia_plant))              # equipment-level CAPEX
fig, ax = plot_stacked_bar(fixed_capital_data(ammonia_plant))             # FCI components

In [ ]:
fig, ax = plot_stacked_bar(variable_opex_data(ammonia_plant))             # feeds and utilities
fig, ax = plot_stacked_bar(fixed_opex_data(ammonia_plant, pct=True))      # fixed OPEX as percentages

In [ ]:
# Where does the levelized cost come from? Discounted CAPEX vs. OPEX (by-product credits would appear as negative bars)
lcop_breakdown = levelized_cost_data(ammonia_plant)
fig, ax = plot_stacked_bar(lcop_breakdown)

Passing a **list of plants** puts one bar per plant side by side. Let us compare the base case with the
cheaper-hydrogen variant from notebook 02:

In [ ]:
from copy import deepcopy

cheap_h2 = deepcopy(ammonia_plant)
cheap_h2.update_configuration({"plant_name": "Cheaper H2", "variable_opex_inputs": {"hydrogen": {"price": 1.50}}})
cheap_h2.calculate_all()

fig, ax = plot_stacked_bar(levelized_cost_data([ammonia_plant, cheap_h2]))
fig, ax = plot_stacked_bar(variable_opex_data([ammonia_plant, cheap_h2]))

## 3) Cumulative cash-flow diagram

The diagram shows the dip into debt during construction, the point of maximum investment, the break-even year,
and the final cumulative position. The returned dictionary carries these numbers so you can quote them directly.

In [ ]:
cash_flow = cash_flow_data([ammonia_plant, cheap_h2])
fig, ax = plot_cash_flow(cash_flow)

for curve in cash_flow["curves"]:
    print(f"{curve['plant']:<20} max investment {curve['max_investment']:>14,.0f} USD in year {curve['max_investment_year']:.0f}; "
          f"payback after {curve['payback_time']:.1f} years")

## 4) One-way sensitivity curves

`sensitivity_data()` varies **one** parameter by ± a fraction of its base value, recalculates the plant at each
point, and returns the curve of the chosen `metric` (`LCOP`, `NPV`, `IRR`, `ROI`, or `PBT`). The parameter can be:

- a price: `"hydrogen"`, `"electricity"`, `"ammonia"` (any `variable_opex_inputs` or `plant_products` key)
- a financial assumption: `"fixed_capital"`, `"fixed_opex"`, `"interest_rate"`, `"project_lifetime"`, `"plant_utilization"`, `"tax_rate"`, `"operator_hourly_rate"`
- a process quantity: `"hydrogen.consumption"`, `"ammonia.production"`

In [ ]:
h2_sensitivity = sensitivity_data(ammonia_plant, parameter="hydrogen", plus_minus_value=0.5, metric="LCOP",
                                  label="Levelized cost of ammonia / [USD/kg]")
fig, ax = plot_sensitivity(h2_sensitivity)

In [ ]:
# Several plants on one figure, and a profitability metric this time
npv_vs_price = sensitivity_data([ammonia_plant, cheap_h2], parameter="ammonia", plus_minus_value=0.3,
                                metric="NPV", n_points=31)
fig, ax = plot_sensitivity(npv_vs_price)

The returned dictionary has one entry per plant under `"curves"`, each with the relative change `x` (in %) and
the metric `y`. With `numpy.interp` we can read the break-even point off the curve:

In [ ]:
import numpy as np

curve = npv_vs_price["curves"][0]                       # base-case plant
x_pct, npv = np.asarray(curve["x"]), np.asarray(curve["y"])
breakeven_pct = np.interp(0.0, npv, x_pct)              # NPV increases with price, so interpolation is valid
base_price = ammonia_plant.plant_products["ammonia"]["price"]
print(f"NPV = 0 at {breakeven_pct:+.1f} % of the base price, i.e. {base_price * (1 + breakeven_pct / 100):.3f} USD/kg")

## 5) Tornado plots

A tornado plot varies every parameter one at a time by the same ± fraction and ranks them by the swing they
cause in the metric. It is the quickest way to see which inputs deserve better data. By default it covers prices
and financial assumptions; `include_process_params=True` adds consumption and production quantities.

In [ ]:
fig, ax = plot_tornado(tornado_data(ammonia_plant, plus_minus_value=0.3, metric="LCOP"))

In [ ]:
fig, ax = plot_tornado(tornado_data(ammonia_plant, plus_minus_value=0.3, metric="NPV", include_process_params=True))

> **A caveat.** Varying `Ammonia prod.` alone while hydrogen and nitrogen consumption stay fixed is not a
> physically reachable state: less ammonia means less hydrogen consumed. Treat process-parameter bars as upper
> bounds on their isolated leverage. Notebook 04 shows how to declare such dependencies so that the sweep
> respects them.

## 6) Task 3.1: at what electricity price does the project stop being profitable?

1. Run a one-way sensitivity of the **NPV** to the **electricity price** over ± 100 % (`plus_minus_value=1.0`, `n_points=41`).
2. Plot it, then use `numpy.interp` on the returned curve to find the electricity price at which NPV = 0.
   Watch the direction: NPV *decreases* with price, so flip the arrays (`npv[::-1]`, `x[::-1]`) before interpolating.
3. Compare with the hydrogen price: run the same analysis for `"hydrogen"` with `plus_minus_value=0.3`. Which of
   the two feeds is the project more exposed to, in relative terms?

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
def breakeven_price(plant, parameter, plus_minus_value, n_points=41):
    data = sensitivity_data(plant, parameter=parameter, plus_minus_value=plus_minus_value, metric="NPV", n_points=n_points)
    curve = data["curves"][0]
    x, npv = np.asarray(curve["x"]), np.asarray(curve["y"])
    pct = np.interp(0.0, npv[::-1], x[::-1])             # NPV falls as an input price rises
    base = plant.variable_opex_inputs[parameter]["price"]
    return data, pct, base * (1 + pct / 100)

for parameter, pm in [("electricity", 1.0), ("hydrogen", 0.3)]:
    data, pct, price = breakeven_price(ammonia_plant, parameter, pm)
    fig, ax = plot_sensitivity(data)
    print(f"{parameter}: NPV = 0 at {pct:+.1f} % of the base price -> {price:.3f} USD/unit")
```

The electricity price has to rise by about 30 % (to roughly 0.10 USD/kWh) before the NPV turns negative, while a
hydrogen price increase of only about 4 % (to roughly 2.08 USD/kg) already does it. In relative terms the project
is far more exposed to the hydrogen price.
</details>

## 7) Task 3.2 (bonus): make it publication-ready

Every plot function accepts `show=False` and returns `(fig, ax)`, so you can restyle it with matplotlib. Take the
variable OPEX comparison of the two plants, give it a title, remove the top and right spines, and save it as a
PNG at 300 dpi with `fig.savefig(...)`.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
fig, ax = plot_stacked_bar(variable_opex_data([ammonia_plant, cheap_h2]), show=False)
ax.set_title("Variable OPEX: base case vs. cheaper hydrogen")
ax.spines[["right", "top"]].set_visible(False)
fig.tight_layout()
fig.savefig("variable_opex_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
```
</details>

## Recap

- `*_data()` returns numbers, `plot_*()` draws them; pass a list of plants to compare.
- The cash-flow diagram gives maximum investment and payback at a glance.
- One-way sensitivity curves answer "at what price does this break even?"; tornado plots rank the drivers.
- Both are deterministic and vary one input at a time. The next notebook varies them all at once.

**Next:** [Exercise 4 - Monte Carlo uncertainty analysis](04_monte_carlo_uncertainty.ipynb)